# CIFAR-10 CNN: HOSVD vs HOOI fine-tuning comparison

## 目的

`04_hooi_tucker2.ipynb` では、同じ balanced rank `(rank_out, rank_in)` に対して HOOI は HOSVD より `conv2.weight` の再構成誤差を小さくできた一方、**fine-tuning 前の分類精度は改善しなかった**。

そこでこの実験では、HOSVD Tucker-2 と HOOI Tucker-2 を **完全に同一の fine-tuning 条件** で学習し、

**HOOI によるより良い weight 初期近似が、fine-tuning 後の回復量・最終精度・収束速度に差を出すか**

を確認する。

この Notebook では rank 探索は行わない。02で選択した balanced rank を使用する。


## 1. 比較する2条件

対象は同じ学習済み baseline の `conv2`。圧縮後のネットワーク構造も同一にする。

### A. HOSVD 初期化

- HOSVD Tucker-2 で `conv2` を分解
- `1×1 -> k×k -> 1×1` の3層へ置換
- 03_finetuning と同じ方法で fine-tuning

### B. HOOI 初期化

- 同じ元 weight・同じ rank から HOSVD で初期化
- HOOI を収束まで反復して factor / core を改善
- Aと全く同じ `1×1 -> k×k -> 1×1` の3層へ置換
- Aと同じ条件で fine-tuning


## 2. 固定する条件

初期化方法以外の差をできるだけ無くす。

- baseline checkpoint: 03 / 04 と同じもの
- data split: train 40k / val_early_stop 5k / val_rank 5k / test 10k
- rank: 02の `selected_rank_settings.csv` から balanced を自動取得
- 圧縮対象: `conv2` のみ
- 圧縮構造: Tucker-2 の `1×1 -> k×k -> 1×1`
- optimizer: Adam
- learning rate: `3e-4`
- max epochs: `30`
- patience: `3`
- min_delta: `1e-4`
- seed: `0`
- Early Stopping: `val_early_stop` のみを使用
- test: 学習・モデル選択には使わず最終評価のみ

各方式の fine-tuning 開始前に seed と DataLoader generator を同じ状態へ戻し、batch 順序や data augmentation の乱数条件も可能な範囲で揃える。


## 3. 記録する指標

fine-tuning 前後を分けて記録する。

### 分解直後

- weight relative Frobenius error
- validation accuracy
- test accuracy

### fine-tuning 後

- best validation accuracy
- test accuracy
- best epoch
- 実行 epoch 数
- validation loss / accuracy の学習曲線

### 派生指標

- `recovery = test_acc_after - test_acc_before`
- `gap_to_baseline = test_acc_after - baseline_test_acc`
- HOOI と HOSVD の最終 test accuracy 差
- 同等精度へ到達するまでの epoch 数の差


## 4. この実験で分かること

### HOOI が fine-tuning 後も高精度

HOOI のより良い低ランク近似が、fine-tuning の初期値としても有利だった可能性がある。

### 最終精度はほぼ同じだが HOOI の収束が速い

最終解は同程度でも、HOOI 初期化が optimizer にとって良い開始点になっている可能性がある。

### HOSVD / HOOI で最終精度がほぼ同じ

fine-tuning によって初期分解法の差がほぼ吸収されたと考えられる。この場合、計算量の小さい HOSVD で十分という実用上の判断につながる。

### HOOI の方が最終精度も悪い

weight reconstruction error の改善は task loss に対して必ずしも良い初期値を意味しない、という結果になる。


## 5. 実装方針

03 と 04 の処理を組み合わせる。

1. 03と同じ baseline / DataLoader / fine-tuning 設定を準備
2. balanced rank を02のCSVから読み込む
3. HOSVD Tucker-2 モデルを baseline から作成
4. HOOI Tucker-2 モデルを **同じ baseline** から別に作成
5. fine-tuning 前の2モデルを評価
6. HOSVDモデルを seed=0 から fine-tuning
7. seed / DataLoader generator をリセット
8. HOOIモデルを同じ条件で fine-tuning
9. 学習曲線と最終結果を同じ表で比較
10. CSV / checkpoint / plot を `05_hosvd_vs_hooi_finetuning` 以下へ保存

> 04で確認済みの `tucker2_hooi_sweep()` / HOOI 更新則そのものを再び学習課題にはしない。ここでの主題は **初期化方法を除いた条件を揃えた fine-tuning 比較**。


## 6. 追加で確認したいこと

1 seed だけで差が小さい場合は、初期化差より学習のばらつきの方が大きい可能性がある。

その場合は `seed = 0, 1, 2` など複数 seed で同じ比較を行い、

- mean test accuracy
- standard deviation
- mean best epoch

まで比較する。

単一seedで明確な差が出なければ、複数seedで再現性を確認してから HOOI 初期化の優劣を判断する。


## 完了条件

- [ ] HOSVD / HOOI が同じ balanced rank・同じ圧縮構造になっている
- [ ] fine-tuning 前の weight error / val / test を保存する
- [ ] 同一 optimizer / LR / seed / Early Stopping 条件で両方を fine-tuning する
- [ ] fine-tuning 後の val / test / best epoch / recovery を比較する
- [ ] 学習曲線を同じ図に重ねて確認する
- [ ] HOOI 初期化の利点が `最終精度` / `回復量` / `収束速度` のどこに現れたか結論を書く
- [ ] 差が小さい場合は複数seed実験の要否を判断する
